In [2]:
import numpy as np 
from polynomial_sampler import *


In [3]:
my_data_raw = np.load("./Points_5d.npy")

my_data_x = np.array([my_data_raw[:,0],
                     my_data_raw[:,1],
                     my_data_raw[:,2],
                     np.exp(my_data_raw[:,3]),
                     my_data_raw[:,4]]).T

In [5]:
sampler = SparsePolynomialSampler(
    data_x=my_data_x,
    max_degree=5,
    num_vars=5,
    max_num_monomials=5,
    prob_add= lambda i,n_iter: 0., 
    prob_remove= lambda i,n_iter: 0., 
    prob_modify= lambda i,n_iter: 0.5, 
    prob_multiply= lambda i,n_iter: 0.25, 
    prob_divide= lambda i,n_iter: 0.25, 
    regularisation_factor=1e3,
    sigma_proposal = lambda i,n_iter: 0.1,
    )

Pré-calcul des transitions multiply/divide...
  Moyenne de transitions multiply par terme: 2.50
  Moyenne de transitions divide par terme: 2.50
Nombre de termes possibles dans le polynôme: 252


In [10]:
all_out = sampler.run_annealing_is(
    n_iter=1000,       
    n_particles=1000,  
    verbose=True,
    adaptative_temp=True,
    degree_bias=0.,
    sparsity_factor=0.,
    adaptation_strength=lambda i,n_iter : 0.05,
    target_acc_rate = lambda x,y : 0.8 - 0.5 * x/y,
    beta_schedule=lambda i,n_iter: 1e-10
)

Itération 1/1000, Beta: 1e-10, Taux d'acceptation: 1.000, ESS: 1000.00/1000, Meilleure perte: 39801.318824, Temps écoulé: 0.93s, probas (0.5, 0.25, 0.25, 0.0, 0.0), best pol : 0.1061*x3*x4^2 + 1.0297*x2*x5^2 - 0.8602*x2^2*x3^2*x4 + 0.2715*x1*x2*x4^2*x5 - 0.2691*x1^2*x3^3
Itération 2/1000, Beta: 1.0500000000000001e-10, Taux d'acceptation: 1.000, ESS: 999.99/1000, Meilleure perte: 35024.684988, Temps écoulé: 1.64s, probas (0.5, 0.25, 0.25, 0.0, 0.0), best pol : 1.0297*x2*x5 + 0.1061*x3*x4^2 - 0.8602*x2^2*x3^2*x4 + 0.2715*x1*x2*x4^2*x5 - 0.2691*x1^2*x3^3
Itération 3/1000, Beta: 1.1025000000000002e-10, Taux d'acceptation: 1.000, ESS: 999.88/1000, Meilleure perte: 35024.684988, Temps écoulé: 2.31s, probas (0.5, 0.25, 0.25, 0.0, 0.0), best pol : 1.1381*x2*x5 + 0.1061*x3*x4^2 - 0.8602*x2^2*x3^2*x4 + 0.2715*x1*x2*x4^2*x5 - 0.2691*x1^2*x3^3
Itération 4/1000, Beta: 1.1576250000000003e-10, Taux d'acceptation: 1.000, ESS: 999.87/1000, Meilleure perte: 29433.131655, Temps écoulé: 3.01s, probas (0.5

In [15]:
best_coeffs = all_out[0]

In [18]:
bc_trunc = sampler.apply_coefficient_threshold(best_coeffs)
print(sampler.evaluate_polynomial_batch(np.array([bc_trunc])))
res, loss = sampler.local_search(best_coeffs,n_steps=1000,use_reg=False)
print(loss)

[7.65868463]
[0.01987722]


In [19]:
sampler.polynomial_to_string(res)
sampler.min_coeff_threshold = 1e-3
res_trunc = sampler.apply_coefficient_threshold(res)
nonzero_indices = np.nonzero(res_trunc)
coeff_min = np.min(abs(res_trunc[nonzero_indices]))
print(sampler.polynomial_to_string(res_trunc/coeff_min))

-1.4198*x2 + 1.4192*x2*x4 - 1.4177*x1*x3 - 1.0000*x1*x4*x5 + x2*x3*x4*x5
